# LeetCode 3873 — Maximum Points Activated with One Addition bằng Python

- Link: https://leetcode.com/problems/maximum-points-activated-with-one-addition/
- Độ khó: **Hard**
- Chủ đề: Thành phần liên thông, Union-Find trên **tọa độ** (đồ thị hai phía hàng–cột), BFS theo nhóm, tham lam chọn hai thành phần lớn nhất

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxActivated(self, points: list[list[int]]) -> int:
```

## 1. Mục tiêu học

1. Mô hình hóa "kích hoạt lan theo cùng `x` hoặc cùng `y`" thành **thành phần liên thông**.
2. Hiểu mô hình **đồ thị hai phía**: mỗi điểm `(x, y)` là một **cạnh** nối đỉnh "cột `x`" với đỉnh "hàng `y`".
3. Chứng minh: điểm thêm vào nối được **tối đa hai** thành phần ⇒ đáp án = hai thành phần lớn nhất + 1.
4. Cài đặt Union-Find với khóa là tọa độ (dùng `dict`, tách không gian `x` và `y`).
5. So sánh với BFS gom nhóm theo `x`/`y` và DSU trên điểm.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho các điểm phân biệt `points[i] = [x, y]`. Khi một điểm được kích hoạt, mọi điểm **cùng hoành độ** hoặc **cùng tung độ** cũng được kích hoạt; lan truyền tới khi không thêm được nữa. Ta được thêm **một** điểm mới ở tọa độ nguyên bất kỳ (chưa có trong `points`) và bắt đầu kích hoạt từ điểm đó. Tìm số điểm được kích hoạt nhiều nhất (**tính cả** điểm mới).

**Ví dụ 1:** `[[1,1],[1,2],[2,2]]` → `4` (thêm `(1,3)`: kích hoạt cả 3 điểm cũ).

**Ví dụ 2:** `[[2,2],[1,1],[3,3]]` → `3` (thêm `(1,2)` nối `(1,1)` và `(2,2)`).

**Ví dụ 3:** `[[2,3],[2,2],[1,1],[4,5]]` → `4` (thêm `(2,1)`).

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `-10^9 ≤ x, y ≤ 10^9`, các điểm phân biệt.

In [1]:
from collections import defaultdict, Counter, deque
import random, time

examples = [([[1, 1], [1, 2], [2, 2]], 4), ([[2, 2], [1, 1], [3, 3]], 3), ([[2, 3], [2, 2], [1, 1], [4, 5]], 4)]
for pts, exp in examples:
    print(pts, "-> kỳ vọng", exp)

[[1, 1], [1, 2], [2, 2]] -> kỳ vọng 4
[[2, 2], [1, 1], [3, 3]] -> kỳ vọng 3
[[2, 3], [2, 2], [1, 1], [4, 5]] -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1. Gom nhóm bằng `defaultdict(list)`

`by_x[x]` = danh sách chỉ số điểm có hoành độ `x`; tương tự `by_y`. Đây là "danh sách kề ngầm": hai điểm kề nhau nếu cùng nhóm.

In [2]:
pts = [[2, 3], [2, 2], [1, 1], [4, 5]]
by_x, by_y = defaultdict(list), defaultdict(list)
for i, (x, y) in enumerate(pts):
    by_x[x].append(i); by_y[y].append(i)
print(dict(by_x), dict(by_y))

{2: [0, 1], 1: [2], 4: [3]} {3: [0], 2: [1], 1: [2], 5: [3]}


### 3.2. Union-Find với khóa tùy ý (`dict`)

Khi khóa là số lớn/âm (tọa độ tới `10^9`), dùng `dict` cho `parent` và `size`, khởi tạo lười khi gặp lần đầu.

In [3]:
class DSU:
    def __init__(self):
        self.parent, self.size = {}, {}
    def find(self, a):
        if a not in self.parent:
            self.parent[a] = a; self.size[a] = 1
        root = a
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[a] != root:
            self.parent[a], a = root, self.parent[a]
        return root
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return False
        if self.size[ra] < self.size[rb]: ra, rb = rb, ra
        self.parent[rb] = ra; self.size[ra] += self.size[rb]
        return True

d = DSU(); d.union(('x', 5), ('y', -7)); d.union(('y', -7), ('x', 9))
print(d.find(('x', 5)) == d.find(('x', 9)), d.find(('x', 1)) == d.find(('x', 5)))

True False


### 3.3. Tách không gian khóa `x` và `y`

Cột `x = 3` và hàng `y = 3` là **hai đỉnh khác nhau**. Có thể dùng tuple `('x', 3)`, `('y', 3)`, hoặc cộng một hằng số lớn cho `y` (ví dụ `y + 3·10^9`) để không trùng với mọi `x ∈ [-10^9, 10^9]`.

In [4]:
M = 3 * 10**9
print("x =", 3, "| y = 3 ->", 3 + M, "| khoảng x:", (-10**9, 10**9), "| khoảng y dịch:", (-10**9 + M, 10**9 + M))

x = 3 | y = 3 -> 3000000003 | khoảng x: (-1000000000, 1000000000) | khoảng y dịch: (2000000000, 4000000000)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Kích hoạt lan theo "cùng cột hoặc cùng hàng" ⇒ các điểm chia thành **nhóm** (thành phần liên thông): kích hoạt một điểm thì cả nhóm sáng.

Điểm mới `(x, y)` "chạm" vào:

- nhóm chứa cột `x` (nếu có điểm cũ cùng hoành độ `x`);
- nhóm chứa hàng `y` (nếu có điểm cũ cùng tung độ `y`).

Vậy nó gộp được **tối đa 2 nhóm**. Để tối đa, chọn `x` thuộc nhóm lớn nhất và `y` thuộc nhóm lớn thứ hai. Điểm `(x, y)` đó **chắc chắn chưa tồn tại** (nếu tồn tại thì cột `x` và hàng `y` đã cùng nhóm).

Nếu chỉ có **một** nhóm: chọn `x` của nhóm đó và một `y` chưa dùng ⇒ `size + 1`.

Tính tay ví dụ 3: nhóm `{(2,3), (2,2)}` (cùng `x = 2`) cỡ 2, `{(1,1)}` cỡ 1, `{(4,5)}` cỡ 1 ⇒ `2 + 1 + 1 = 4` ✔.

## 5. Cách 1 — Brute Force: thử mọi điểm mới hợp lý + mô phỏng lan truyền

Chỉ cần thử `x` trong tập hoành độ đã có (hoặc một `x` mới), `y` trong tập tung độ đã có (hoặc `y` mới), bỏ các tọa độ đã tồn tại; mô phỏng BFS kích hoạt cho mỗi ứng viên. `O(n³)` — chỉ cho `n` nhỏ.

In [5]:
def simulate(points, start):
    allp = points + [start]
    by_x, by_y = defaultdict(list), defaultdict(list)
    for i, (x, y) in enumerate(allp):
        by_x[x].append(i); by_y[y].append(i)
    seen = {len(allp) - 1}
    dq = deque([len(allp) - 1])
    while dq:
        i = dq.popleft()
        x, y = allp[i]
        for j in by_x[x] + by_y[y]:
            if j not in seen:
                seen.add(j); dq.append(j)
    return len(seen)

def brute(points):
    xs = {x for x, _ in points} | {10**9 + 7}
    ys = {y for _, y in points} | {10**9 + 7}
    existing = {tuple(p) for p in points}
    best = 0
    for x in xs:
        for y in ys:
            if (x, y) not in existing:
                best = max(best, simulate([list(p) for p in points], [x, y]))
    return best

for pts, exp in examples:
    print(brute(pts), "kỳ vọng", exp)

4 kỳ vọng 4
3 kỳ vọng 3
4 kỳ vọng 4


**Nút thắt:** `O(n²)` ứng viên × `O(n)` mô phỏng. Nhưng mọi ứng viên chỉ khác nhau ở **hai nhóm** mà nó chạm ⇒ chỉ cần biết **kích thước các nhóm**.

## 6. Ý tưởng thuật toán / bước đột phá

**Mô hình hai phía:** đỉnh là các **cột** `x` và các **hàng** `y`; mỗi điểm `(x, y)` là một **cạnh** nối cột `x` với hàng `y`. Hai điểm cùng nhóm kích hoạt ⇔ hai cạnh nằm trong cùng thành phần liên thông của đồ thị hai phía. Kích thước nhóm = **số cạnh** (số điểm) trong thành phần.

**Kết luận:** đáp án = `s₁ + s₂ + 1`, với `s₁ ≥ s₂` là số điểm của hai thành phần lớn nhất (`s₂ = 0` nếu chỉ có một thành phần).

**Vì sao không làm tốt hơn được:** điểm mới chạm tối đa một cột và một hàng ⇒ tối đa hai thành phần; điểm mới không làm thay đổi các thành phần khác.

**Hình dung đồ thị hai phía cho ví dụ 3.** Bên trái là các cột `C1, C2, C4`, bên phải là các hàng `R1, R2, R3, R5`. Mỗi điểm là một cạnh: `(2,3)` = `C2–R3`, `(2,2)` = `C2–R2`, `(1,1)` = `C1–R1`, `(4,5)` = `C4–R5`. Ba thành phần: `{C2, R2, R3}` (2 cạnh), `{C1, R1}` (1 cạnh), `{C4, R5}` (1 cạnh). Điểm mới `(2, 1)` là cạnh `C2–R1` nối hai thành phần đầu ⇒ `2 + 1 + 1 = 4`.

Lưu ý: số **điểm** của thành phần = số **cạnh**, không phải số đỉnh. Một thành phần có `c` cột và `r` hàng có từ `c + r - 1` tới `c · r` điểm.

In [6]:
pts = [[2, 3], [2, 2], [1, 1], [4, 5]]
comp = defaultdict(list)
d = DSU()
for x, y in pts:
    d.union(('C', x), ('R', y))
for x, y in pts:
    comp[d.find(('C', x))].append(f"C{x}–R{y}")
for i, edges in enumerate(comp.values()):
    print(f"thành phần {i}: cạnh {edges} -> {len(edges)} điểm")

thành phần 0: cạnh ['C2–R3', 'C2–R2'] -> 2 điểm
thành phần 1: cạnh ['C1–R1'] -> 1 điểm
thành phần 2: cạnh ['C4–R5'] -> 1 điểm


## 7. Các cách tiếp theo

- **Cách 2:** BFS trên các **điểm**, láng giềng lấy từ `by_x`, `by_y` (đánh dấu nhóm đã duyệt để không lặp) — `O(n)`.
- **Cách 3:** **Union-Find trên tọa độ**: `union(cột x, hàng y)` cho mỗi điểm, đếm số điểm theo gốc — `O(n α)` (lời giải nộp chính).
- **Cách 4:** **Union-Find trên điểm**: gộp các điểm liên tiếp trong cùng `by_x[x]` và trong cùng `by_y[y]` — `O(n α)`.

### 7.1. Cách 2 — BFS gom nhóm

**Mẹo quan trọng:** khi đã duyệt hết một cột `x` (hoặc hàng `y`), đánh dấu nó để không duyệt lại danh sách đó — tránh `O(n²)` khi một cột có rất nhiều điểm.

In [7]:
def top_two_plus_one(sizes):
    s = sorted(sizes, reverse=True) + [0]
    return s[0] + s[1] + 1

def bfs_components(points):
    n = len(points)
    by_x, by_y = defaultdict(list), defaultdict(list)
    for i, (x, y) in enumerate(points):
        by_x[x].append(i); by_y[y].append(i)
    seen = [False] * n
    used_x, used_y = set(), set()
    sizes = []
    for s in range(n):
        if seen[s]:
            continue
        seen[s] = True
        dq = deque([s]); cnt = 0
        while dq:
            i = dq.popleft(); cnt += 1
            x, y = points[i]
            if x not in used_x:
                used_x.add(x)
                for j in by_x[x]:
                    if not seen[j]:
                        seen[j] = True; dq.append(j)
            if y not in used_y:
                used_y.add(y)
                for j in by_y[y]:
                    if not seen[j]:
                        seen[j] = True; dq.append(j)
        sizes.append(cnt)
    return top_two_plus_one(sizes)

for pts, exp in examples:
    print(bfs_components(pts), "kỳ vọng", exp)

4 kỳ vọng 4
3 kỳ vọng 3
4 kỳ vọng 4


**Độ phức tạp:** mỗi điểm vào hàng đợi một lần, mỗi danh sách `by_x[x]`/`by_y[y]` duyệt một lần ⇒ `O(n)`.

### 7.2. Cách 3 — Union-Find trên tọa độ (lời giải chính)

**Giải thích từng đoạn:**

1. Với mỗi điểm `(x, y)`: `union(x, y + M)` với `M = 3·10^9` (tách không gian).
2. Đếm số điểm theo gốc: `cnt[find(x)] += 1`.
3. Lấy hai giá trị lớn nhất của `cnt` + 1.

In [8]:
def dsu_coords(points):
    M = 3 * 10**9
    d = DSU()
    for x, y in points:
        d.union(x, y + M)
    cnt = Counter(d.find(x) for x, _ in points)
    return top_two_plus_one(cnt.values())

for pts, exp in examples:
    print(dsu_coords(pts), "kỳ vọng", exp)

4 kỳ vọng 4
3 kỳ vọng 3
4 kỳ vọng 4


**Độ phức tạp Cách 3:** `O(n α(n))` thời gian, `O(n)` bộ nhớ (tối đa `2n` đỉnh tọa độ).

### 7.3. Cách 4 — Union-Find trên điểm

Trong mỗi nhóm `by_x[x]`, gộp các điểm liên tiếp (chỉ cần `|nhóm| - 1` phép gộp). Tương tự cho `by_y`. Kích thước thành phần = số điểm.

In [9]:
def dsu_points(points):
    n = len(points)
    parent = list(range(n)); size = [1] * n
    def find(a):
        root = a
        while parent[root] != root:
            root = parent[root]
        while parent[a] != root:
            parent[a], a = root, parent[a]
        return root
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb: return
        if size[ra] < size[rb]: ra, rb = rb, ra
        parent[rb] = ra; size[ra] += size[rb]
    by_x, by_y = defaultdict(list), defaultdict(list)
    for i, (x, y) in enumerate(points):
        by_x[x].append(i); by_y[y].append(i)
    for group in list(by_x.values()) + list(by_y.values()):
        for a, b in zip(group, group[1:]):
            union(a, b)
    sizes = [size[i] for i in range(n) if find(i) == i]
    return top_two_plus_one(sizes)

for pts, exp in examples:
    print(dsu_points(pts), "kỳ vọng", exp)

4 kỳ vọng 4
3 kỳ vọng 3
4 kỳ vọng 4


In [10]:
class SolutionPoints:
    def maxActivated(self, points: list[list[int]]) -> int:
        return dsu_points(points)

print([SolutionPoints().maxActivated(p) for p, _ in examples])

[4, 3, 4]


**Độ phức tạp Cách 4:** `O(n α(n))`, bộ nhớ `O(n)`.

## 8. TRACE từng bước

Trace Cách 3 trên `points = [[1,1],[1,2],[2,2],[5,7],[6,7],[9,9]]` (tự dựng). Ký hiệu cột `Cx`, hàng `Ry`.

| điểm | union | thành phần sau khi gộp |
|---|---|---|
| (1,1) | C1–R1 | {C1, R1} |
| (1,2) | C1–R2 | {C1, R1, R2} |
| (2,2) | C2–R2 | {C1, C2, R1, R2} |
| (5,7) | C5–R7 | {C5, R7} |
| (6,7) | C6–R7 | {C5, C6, R7} |
| (9,9) | C9–R9 | {C9, R9} |

Số điểm theo thành phần: `3, 2, 1` ⇒ đáp án `3 + 2 + 1 = 6` (ví dụ thêm điểm `(1, 7)`).

In [11]:
def trace(points):
    M = 3 * 10**9; d = DSU()
    name = lambda k: f"C{k}" if k < M // 2 else f"R{k - M}"
    for x, y in points:
        d.union(x, y + M)
        comp = sorted(name(k) for k in d.parent if d.find(k) == d.find(x))
        print(f"điểm ({x},{y}): union C{x}–R{y} -> thành phần {comp}")
    cnt = Counter(d.find(x) for x, _ in points)
    print("số điểm theo thành phần:", sorted(cnt.values(), reverse=True), "-> đáp án", top_two_plus_one(cnt.values()))

pts = [[1, 1], [1, 2], [2, 2], [5, 7], [6, 7], [9, 9]]
trace(pts)
print("kiểm chứng: thêm (1, 7) kích hoạt", simulate(pts, [1, 7]), "điểm | vét cạn:", brute(pts))

điểm (1,1): union C1–R1 -> thành phần ['C1', 'R1']
điểm (1,2): union C1–R2 -> thành phần ['C1', 'R1', 'R2']
điểm (2,2): union C2–R2 -> thành phần ['C1', 'C2', 'R1', 'R2']
điểm (5,7): union C5–R7 -> thành phần ['C5', 'R7']
điểm (6,7): union C6–R7 -> thành phần ['C5', 'C6', 'R7']
điểm (9,9): union C9–R9 -> thành phần ['C9', 'R9']
số điểm theo thành phần: [3, 2, 1] -> đáp án 6
kiểm chứng: thêm (1, 7) kích hoạt 6 điểm | vét cạn: 6


## 9. Code nộp LeetCode

In [12]:
from collections import Counter

class Solution:
    def maxActivated(self, points: list[list[int]]) -> int:
        M = 3 * 10**9
        parent, size = {}, {}

        def find(a):
            if a not in parent:
                parent[a] = a
                size[a] = 1
                return a
            root = a
            while parent[root] != root:
                root = parent[root]
            while parent[a] != root:
                parent[a], a = root, parent[a]
            return root

        for x, y in points:
            ra, rb = find(x), find(y + M)
            if ra != rb:
                if size[ra] < size[rb]:
                    ra, rb = rb, ra
                parent[rb] = ra
                size[ra] += size[rb]

        cnt = Counter(find(x) for x, _ in points)
        top1 = top2 = 0
        for c in cnt.values():
            if c > top1:
                top1, top2 = c, top1
            elif c > top2:
                top2 = c
        return top1 + top2 + 1

sol = Solution()
for pts, exp in examples:
    got = sol.maxActivated(pts)
    print(got, "✔" if got == exp else "✘")

4 ✔
3 ✔
4 ✔


## 10. Vì sao thuật toán đúng

1. **Thành phần = nhóm kích hoạt:** kích hoạt lan qua cạnh "cùng cột/cùng hàng"; trong đồ thị hai phía, hai điểm (cạnh) cùng thành phần ⇔ có dãy điểm nối nhau bằng cột/hàng chung ⇔ kích hoạt lan từ điểm này tới điểm kia.
2. **Cận trên:** điểm mới `(x, y)` là một cạnh mới nối cột `x` và hàng `y`. Thành phần chứa nó gồm (tối đa) thành phần cũ của cột `x` và thành phần cũ của hàng `y` ⇒ số điểm kích hoạt ≤ `s₁ + s₂ + 1`.
3. **Đạt được:** lấy cột `x` của thành phần lớn nhất `A` và hàng `y` của thành phần lớn thứ hai `B ≠ A`. `(x, y)` chưa tồn tại vì nếu có thì `x`, `y` cùng thành phần. Kích hoạt `(x, y)` lan ra cả `A` và `B` ⇒ `s₁ + s₂ + 1`. Nếu chỉ có một thành phần, chọn `y` chưa xuất hiện ⇒ `s₁ + 1`.
4. **Đếm đúng:** mỗi điểm được đếm đúng một lần vào gốc của cột `x` của nó.

## 11. Phân tích Time / Space Complexity

Đặt `n = len(points)`.

| Cách | Time | Space |
|---|---|---|
| 1. Thử ứng viên + mô phỏng | `O(n³)` | `O(n)` |
| 2. BFS gom nhóm | `O(n)` | `O(n)` |
| 3. DSU trên tọa độ | `O(n α(n))` | `O(n)` |
| 4. DSU trên điểm | `O(n α(n))` | `O(n)` |

## 12. Bộ test

In [13]:
def run_all(pts, with_brute=True):
    res = [bfs_components(pts), dsu_coords(pts), dsu_points(pts), Solution().maxActivated(pts)]
    if with_brute:
        res.append(brute(pts))
    return res

for pts, exp in examples:
    assert all(v == exp for v in run_all(pts))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [14]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng vét cạn)
cases = [
    ([[0, 0]], 2),                                     # 1 điểm
    ([[0, 0], [1, 1]], 3),                             # 2 nhóm cỡ 1
    ([[0, 0], [0, 1], [0, 2]], 4),                     # 1 nhóm cỡ 3
    ([[-10**9, -10**9], [10**9, 10**9]], 3),           # tọa độ biên
    ([[3, 3], [3, 5], [5, 3], [7, 7], [8, 8]], 5),     # x và y cùng giá trị số nhưng khác "không gian"
]
for pts, exp in cases:
    r = run_all(pts)
    assert all(v == exp for v in r), (pts, r)
print("Ca biên: OK")

Ca biên: OK


In [15]:
# 12.3 phản ví dụ: KHÔNG tách không gian x và y (dùng chung khóa) gây gộp nhầm
def wrong_no_shift(points):
    d = DSU()
    for x, y in points:
        d.union(x, y)                                 # SAI: cột 2 và hàng 2 bị coi là một
    cnt = Counter(d.find(x) for x, _ in points)
    return top_two_plus_one(cnt.values())

pts = [[1, 2], [2, 5], [7, 7]]   # cột 2 (điểm thứ 2) và hàng 2 (điểm thứ 1) KHÔNG liên quan
print("sai:", wrong_no_shift(pts), "| đúng:", Solution().maxActivated(pts), "| vét cạn:", brute(pts))

sai: 4 | đúng: 3 | vét cạn: 3


In [16]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3873)
for _ in range(300):
    n = rng.randint(1, 8)
    pts = set()
    while len(pts) < n:
        pts.add((rng.randint(0, 5), rng.randint(0, 5)))
    pts = [list(p) for p in pts]
    r = run_all(pts)
    assert len(set(r)) == 1, (pts, r)
print("300 test ngẫu nhiên (có vét cạn): OK")

300 test ngẫu nhiên (có vét cạn): OK


In [17]:
# 12.5 test lớn đúng ràng buộc
rng = random.Random(9)
pts = set()
while len(pts) < 10**5:
    pts.add((rng.randint(-10**9, 10**9) % 50000, rng.randint(-10**9, 10**9) % 50000))
pts = [list(p) for p in pts]
t0 = time.time(); a = Solution().maxActivated(pts); t1 = time.time()
print("đáp án:", a, "| khớp BFS:", a == bfs_components(pts), "| khớp DSU điểm:", a == dsu_points(pts), "| thời gian < 2s:", t1 - t0 < 2)
col = [[0, y] for y in range(10**5)]
print("một cột 10^5 điểm:", Solution().maxActivated(col), "| BFS:", bfs_components(col))

đáp án: 95798 | khớp BFS: True | khớp DSU điểm: True | thời gian < 2s: True
một cột 10^5 điểm: 100001 | BFS: 100001


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không tách không gian `x` và `y`

Đã thấy ở 12.3: `union(x, y)` khiến cột `2` và hàng `2` thành một đỉnh, gộp nhầm hai nhóm không liên quan. **Sửa:** `y + M` hoặc tuple `('y', y)`.

### Lỗi 2 — Trả về `s₁ + s₂` (quên tính điểm mới)

Đề yêu cầu **tính cả** điểm vừa thêm.

In [18]:
def wrong_no_plus_one(points):
    M = 3 * 10**9; d = DSU()
    for x, y in points: d.union(x, y + M)
    s = sorted(Counter(d.find(x) for x, _ in points).values(), reverse=True) + [0]
    return s[0] + s[1]                                 # SAI: thiếu + 1

print("sai:", wrong_no_plus_one(examples[0][0]), "| đúng:", Solution().maxActivated(examples[0][0]))

sai: 3 | đúng: 4


### Lỗi 3 — Đếm kích thước thành phần theo **số đỉnh tọa độ** thay vì **số điểm**

`size` trong DSU tọa độ là số cột + số hàng, không phải số điểm. Ví dụ `[[1,1],[1,2],[2,1],[2,2]]` có 4 điểm nhưng chỉ 4 đỉnh tọa độ (C1, C2, R1, R2) — trùng hợp; với `[[1,1],[1,2],[1,3]]` có 3 điểm nhưng 4 đỉnh.

In [19]:
def wrong_vertex_size(points):
    M = 3 * 10**9; d = DSU()
    for x, y in points: d.union(x, y + M)
    roots = {d.find(k) for k in list(d.parent)}
    s = sorted((d.size[r] for r in roots), reverse=True) + [0]
    return s[0] + s[1] + 1                              # SAI: size = số đỉnh tọa độ

pts = [[1, 1], [1, 2], [1, 3]]
print("sai:", wrong_vertex_size(pts), "| đúng:", Solution().maxActivated(pts))

sai: 5 | đúng: 4


### Lỗi 4 — BFS không đánh dấu cột/hàng đã duyệt

Nếu mỗi lần lấy một điểm lại duyệt **toàn bộ** `by_x[x]`, một cột có `10^5` điểm khiến BFS tốn `10^{10}` bước. **Sửa:** tập `used_x`, `used_y` (như Cách 2).

In [20]:
def bfs_no_mark_ops(points):
    by_x, by_y = defaultdict(list), defaultdict(list)
    for i, (x, y) in enumerate(points):
        by_x[x].append(i); by_y[y].append(i)
    seen = {0}; dq = deque([0]); ops = 0
    while dq:
        i = dq.popleft(); x, y = points[i]
        for j in by_x[x] + by_y[y]:
            ops += 1
            if j not in seen: seen.add(j); dq.append(j)
    return ops

col = [[0, y] for y in range(2000)]
print("số bước khi không đánh dấu (2000 điểm cùng cột):", bfs_no_mark_ops(col), "~ n²")

số bước khi không đánh dấu (2000 điểm cùng cột): 4002000 ~ n²


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Cùng hàng hoặc cùng cột thì liên thông"** ⇒ đồ thị hai phía hàng–cột, mỗi điểm là một cạnh (LC 947 *Most Stones Removed with Same Row or Column*).
- **Thêm một phần tử nối các thành phần** ⇒ đếm kích thước thành phần, chọn những cái lớn nhất mà phần tử mới chạm được (ở đây tối đa 2).
- **Khóa lớn/âm** ⇒ DSU với `dict`; **hai loại đỉnh** ⇒ tách không gian khóa.

## 15. Biến thể của bài

1. **Thêm `k` điểm** (mỗi điểm mới kích hoạt riêng, hoặc cùng lúc): mỗi điểm mới nối tối đa 2 thành phần; `k` điểm mới có thể tạo chuỗi nối `k + 1` thành phần ⇒ tổng `k + 1` thành phần lớn nhất + `k`.
2. **Kích hoạt theo đường chéo** (cùng `x - y` hoặc `x + y`): đổi khóa thành đường chéo.
3. **Không được thêm, chỉ chọn một điểm có sẵn để bắt đầu**: đáp án = thành phần lớn nhất.
4. **LC 947**: số điểm xóa được tối đa = `n - số thành phần`.

In [21]:
# Biến thể 4 (LC 947): số điểm xóa tối đa = n - số thành phần
def most_stones_removed(points):
    M = 3 * 10**9; d = DSU()
    for x, y in points: d.union(x, y + M)
    return len(points) - len({d.find(x) for x, _ in points})

print(most_stones_removed([[0, 0], [0, 1], [1, 0], [1, 2], [2, 1], [2, 2]]), "(LC 947 ví dụ 1 cho 5)")

5 (LC 947 ví dụ 1 cho 5)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử + mô phỏng | Thử mọi điểm mới | `O(n³)` | `O(n)` | Kiểm thử |
| 2. BFS gom nhóm | Duyệt nhóm theo `by_x`, `by_y` | `O(n)` | `O(n)` | Khi quen BFS |
| 3. DSU tọa độ | `union(x, y+M)` | `O(n α)` | `O(n)` | **Lời giải chuẩn**, code ngắn |
| 4. DSU điểm | Gộp điểm liên tiếp trong nhóm | `O(n α)` | `O(n)` | Khi cần thông tin theo điểm |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `maxActivated([[0,0],[5,5],[9,9],[0,9]])` = ?

**Bài 2 (tính tay).** `[[1,1],[2,1],[3,2],[3,3],[4,4]]` — tìm các thành phần và đáp án.

**Bài 3 (điền TODO).** Viết `component_sizes(points)` trả về danh sách kích thước (số điểm) các thành phần, sắp giảm dần.

**Bài 4 (tự viết lại).** Viết Cách 3 dùng tuple `('x', x)`, `('y', y)` làm khóa thay vì dịch `M`.

**Bài 5 (mini challenge).** Biến thể: được thêm **hai** điểm mới, kích hoạt bắt đầu từ **điểm mới thứ nhất**. Tính số điểm kích hoạt tối đa (tính cả hai điểm mới nếu được kích hoạt). Kiểm chứng bằng vét cạn nhỏ.

In [22]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def component_sizes(points):
    # TODO: DSU tọa độ, Counter theo find(x), sắp giảm dần
    pass

In [23]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def max_with_two_additions(points):
    # TODO: hai điểm mới tạo chuỗi nối tối đa 3 thành phần -> s1 + s2 + s3 + 2
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `(0,0)` và `(0,9)` cùng cột 0; `(0,9)` và `(9,9)` cùng hàng 9 ⇒ nhóm cỡ 3. (2) `(5,5)` cỡ 1. (3) `3 + 1 + 1 = 5`.

**Bài 2.** (1) `(1,1)`–`(2,1)` cùng hàng 1. (2) `(3,2)`–`(3,3)` cùng cột 3. (3) `(4,4)` riêng ⇒ cỡ `2, 2, 1` ⇒ đáp án `5`.

**Bài 3.** (1) Như Cách 3. (2) `sorted(cnt.values(), reverse=True)`. (3) Tổng = `n`.

**Bài 4.** (1) `d.union(('x', x), ('y', y))`. (2) Đếm theo `d.find(('x', x))`. (3) So với `Solution`.

**Bài 5.** (1) Điểm mới thứ nhất `(xA, yB)` nối A và B; điểm mới thứ hai `(xB', yC)`… cần chung cột/hàng với điểm mới thứ nhất hoặc nhóm đã nối. (2) Có thể nối thêm một nhóm C ⇒ `s₁ + s₂ + s₃ + 2`. (3) Nếu số nhóm ít hơn, phần thiếu bằng 0 — vẫn cộng 2 điểm mới (điểm thứ hai luôn đặt được cùng cột với điểm thứ nhất).

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().maxActivated([[0, 0], [5, 5], [9, 9], [0, 9]]) == 5 == brute([[0, 0], [5, 5], [9, 9], [0, 9]])
assert Solution().maxActivated([[1, 1], [2, 1], [3, 2], [3, 3], [4, 4]]) == 5
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [25]:
# Bài 3, 4
def component_sizes_sol(points):
    M = 3 * 10**9; d = DSU()
    for x, y in points: d.union(x, y + M)
    return sorted(Counter(d.find(x) for x, _ in points).values(), reverse=True)

def sol_tuple_keys(points):
    d = DSU()
    for x, y in points: d.union(('x', x), ('y', y))
    return top_two_plus_one(Counter(d.find(('x', x)) for x, _ in points).values())

assert component_sizes_sol([[1, 1], [2, 1], [3, 2], [3, 3], [4, 4]]) == [2, 2, 1]
rng = random.Random(4)
for _ in range(200):
    pts = list({(rng.randint(0, 6), rng.randint(0, 6)) for _ in range(rng.randint(1, 12))})
    pts = [list(p) for p in pts]
    assert sol_tuple_keys(pts) == Solution().maxActivated(pts)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [26]:
# Bài 5: hai điểm mới, kích hoạt từ điểm mới thứ nhất
def max_two_sol(points):
    s = component_sizes_sol(points) + [0, 0]
    return s[0] + s[1] + s[2] + 2

def brute_two(points):
    xs = sorted({x for x, _ in points} | {100, 101}); ys = sorted({y for _, y in points} | {100, 101})
    existing = {tuple(p) for p in points}; best = 0
    cands = [(x, y) for x in xs for y in ys if (x, y) not in existing]
    for a in cands:
        for b in cands:
            if a != b:
                allp = points + [list(a), list(b)]
                # mô phỏng từ điểm a
                by_x, by_y = defaultdict(list), defaultdict(list)
                for i, (x, y) in enumerate(allp): by_x[x].append(i); by_y[y].append(i)
                start = len(points); seen = {start}; dq = deque([start])
                while dq:
                    i = dq.popleft(); x, y = allp[i]
                    for j in by_x[x] + by_y[y]:
                        if j not in seen: seen.add(j); dq.append(j)
                best = max(best, len(seen))
    return best

rng = random.Random(5)
for _ in range(60):
    pts = [list(p) for p in {(rng.randint(0, 4), rng.randint(0, 4)) for _ in range(rng.randint(1, 5))}]
    assert max_two_sol(pts) == brute_two(pts), pts
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
M = 3 * 10**9                     # tách không gian hàng/cột
for x, y in points: union(x, y + M)
cnt = Counter(find(x) for x, _ in points)   # số ĐIỂM theo thành phần
top1, top2 = hai giá trị lớn nhất (top2 = 0 nếu chỉ một)
return top1 + top2 + 1
```

**Checklist:**

- [ ] Tách khóa cột và hàng (không để cột `k` trùng hàng `k`)?
- [ ] Đếm **số điểm** theo thành phần (không phải số đỉnh tọa độ)?
- [ ] Cộng `1` cho điểm mới?
- [ ] Trường hợp chỉ một thành phần: `top2 = 0`?
- [ ] `find` không đệ quy sâu (dùng vòng lặp)?